### Importación

In [2]:
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import time
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize, LabelEncoder, StandardScaler
from sklearn.metrics import roc_curve, auc, ConfusionMatrixDisplay
from sklearn.multiclass import OneVsRestClassifier
import matplotlib.pyplot as plt

In [3]:
def cargar_mismas_imagenes(
    carpeta="images_training_rev1",
    archivo_clasificaciones="training_solutions_rev1.csv",
    max_imagenes=12000,
    size=(128, 128)
):
    # Leer CSV
    df = pd.read_csv(archivo_clasificaciones)
    if "GalaxyID" not in df.columns:
        raise ValueError("El archivo CSV debe tener una columna 'GalaxyID'.")
    galaxy_ids = df["GalaxyID"].astype(str).tolist()
    archivos = [f for f in os.listdir(carpeta) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
    archivos_validos = [f for f in archivos if f.split(".")[0] in galaxy_ids]
    # Seleccionar una sola vez
    if len(archivos_validos) > max_imagenes:
        archivos_validos = random.sample(archivos_validos, max_imagenes)
    print(f"Se seleccionaron {len(archivos_validos)} imágenes para ambos conjuntos.")

    # Cargar en gris
    X_gray, y_gray = [], []
    for nombre in archivos_validos:
        galaxy_id = nombre.split(".")[0]
        fila = df[df["GalaxyID"] == int(galaxy_id)].iloc[0]
        etiqueta = fila.drop("GalaxyID").idxmax()
        ruta = os.path.join(carpeta, nombre)
        try:
            img = Image.open(ruta).convert("L").resize(size)
            X_gray.append(np.array(img))
            y_gray.append(etiqueta)
        except Exception as e:
            print(f"Error al cargar {nombre} (gris): {e}")

    # Cargar en RGB
    X_rgb, y_rgb = [], []
    for nombre in archivos_validos:
        galaxy_id = nombre.split(".")[0]
        fila = df[df["GalaxyID"] == int(galaxy_id)].iloc[0]
        etiqueta = fila.drop("GalaxyID").idxmax()
        ruta = os.path.join(carpeta, nombre)
        try:
            img = Image.open(ruta).convert("RGB").resize(size)
            X_rgb.append(np.array(img))
            y_rgb.append(etiqueta)
        except Exception as e:
            print(f"Error al cargar {nombre} (RGB): {e}")

    X_gray = np.array(X_gray)
    X_rgb = np.array(X_rgb)
    print(f"Se cargaron {len(X_gray)} imágenes en gris y {len(X_rgb)} en RGB.")
    return X_gray, y_gray, X_rgb, y_rgb

# Uso:
X_gray, y_gray, X_rgb, y_rgb = cargar_mismas_imagenes(
    carpeta="images_training_rev1",
    archivo_clasificaciones="training_solutions_rev1.csv",
    max_imagenes=12000,
    size=(128, 128)
)

Se seleccionaron 12000 imágenes para ambos conjuntos.
Se cargaron 12000 imágenes en gris y 12000 en RGB.


In [4]:
# --- Preparación: Escala de Grises ---
# Aplanar imágenes
X_gray_flat = X_gray.reshape((X_gray.shape[0], -1))

# Codificar etiquetas
le_gray = LabelEncoder()
y_gray_enc = le_gray.fit_transform(y_gray)

# División 70-30
Xtr_gray, Xte_gray, ytr_gray, yte_gray = train_test_split(
    X_gray_flat, y_gray_enc, test_size=0.3, random_state=42, stratify=y_gray_enc
)

# Escalado
scaler_gray = StandardScaler()
Xtr_gray_scaled = scaler_gray.fit_transform(Xtr_gray)
Xte_gray_scaled = scaler_gray.transform(Xte_gray)

print(f"Gray: Xtr={Xtr_gray_scaled.shape}, Xte={Xte_gray_scaled.shape}, ytr={ytr_gray.shape}, yte={yte_gray.shape}")

Gray: Xtr=(8400, 16384), Xte=(3600, 16384), ytr=(8400,), yte=(3600,)


In [5]:
# --- Preparación: Imágenes RGB ---
# Aplanar imágenes RGB (N, 128, 128, 3) -> (N, 49152)
X_rgb_flat = X_rgb.reshape(X_rgb.shape[0], -1)

# Codificar etiquetas
le_rgb = LabelEncoder()
y_rgb_enc = le_rgb.fit_transform(y_rgb)

# División 70-30
Xtr_rgb, Xte_rgb, ytr_rgb, yte_rgb = train_test_split(
    X_rgb_flat, y_rgb_enc, test_size=0.3, random_state=42, stratify=y_rgb_enc
)

# Escalado
scaler_rgb = StandardScaler()
Xtr_rgb_scaled = scaler_rgb.fit_transform(Xtr_rgb)
Xte_rgb_scaled = scaler_rgb.transform(Xte_rgb)

print(f"RGB: Xtr={Xtr_rgb_scaled.shape}, Xte={Xte_rgb_scaled.shape}, ytr={ytr_rgb.shape}, yte={yte_rgb.shape}")

RGB: Xtr=(8400, 49152), Xte=(3600, 49152), ytr=(8400,), yte=(3600,)


In [ ]:
# Definir y entrenar el modelo SVM One-vs-Rest con kernel RBF
svm_ovr_gray = OneVsRestClassifier(SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=0))

t0 = time.time()
svm_ovr_gray.fit(Xtr_gray_scaled, ytr_gray)
train_time = time.time() - t0
print(f"SVM Gray entrenado en {train_time:.2f}s")

# Predicción y métricas
y_pred_gray = svm_ovr_gray.predict(Xte_gray_scaled)
acc_gray = accuracy_score(yte_gray, y_pred_gray)
print(f"Accuracy SVM Gray: {acc_gray:.4f}")
print("Classification report (Gray):")
print(classification_report(yte_gray, y_pred_gray))

# Matriz de confusión
cm_gray = confusion_matrix(yte_gray, y_pred_gray)
plt.figure(figsize=(8,6))
sns.heatmap(cm_gray, cmap='Blues', fmt='d')
plt.title("Matriz de confusión - SVM (Gray)")
plt.xlabel("Predicción")
plt.ylabel("Verdadero")
plt.show()

# ROC-AUC multiclass
n_classes = len(np.unique(yte_gray))
yte_bin_gray = label_binarize(yte_gray, classes=np.arange(n_classes))
try:
    y_score_gray = svm_ovr_gray.predict_proba(Xte_gray_scaled)
    auc_per_class_gray = []
    for i in range(n_classes):
        try:
            auc_per_class_gray.append(roc_auc_score(yte_bin_gray[:, i], y_score_gray[:, i]))
        except ValueError:
            auc_per_class_gray.append(np.nan)
    print("ROC AUC (todas las clases, Gray):", np.round(auc_per_class_gray,3))
    print("ROC AUC macro (promedio ignorando NaN, Gray):", np.nanmean(auc_per_class_gray))
except Exception as e:
    print(f"ROC AUC no disponible (Gray): {e}")